# Part 4 · Notebook 05 — Connections that survive the real world

**Sessions:** S9 (Connection management) · [Lesson plan](../../docs/lessons/PART_04_BROKER_CONNECTIVITY.md) · graded labs in [`labs/part04/`](../../labs/part04/)

**You will:**
1. Compute reconnect delays with exponential back-off and full jitter.
2. Watch 500 clients hammer a restarted gateway, with and without jitter.
3. Write the circuit breaker that stops you retrying a broker that is down.
4. Sort IB's message codes into what to ignore, what to wait out and what to stop for.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Nothing here connects to a broker: the account rows, bars, ticks and order events are synthetic, shaped like what `ib_async` and `alpaca-py` return.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p4lib.py is in notebooks/part04/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p4lib as p

p.use_course_style()
import random

## 1. Back-off with full jitter

After a disconnect, retry after a delay that doubles each attempt, capped: `min(cap, base · 2**i)` for attempt `i = 0, 1, 2, …`. **Full jitter** waits a random time between 0 and that ceiling: `rng.uniform(0, min(cap, base * 2**i))`. Call `rng.uniform` once per attempt, in order, so the result is reproducible for a given seed.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def backoff_delays(attempts: int, base: float, cap: float, rng: random.Random) -> list[float]:
    return [... for i in range(attempts)]        # ✍️ full jitter for attempt i

mine = backoff_delays(8, 1.0, 30.0, random.Random(42))
mine = p.check("backoff_delays", mine, p.backoff_delays(8, 1.0, 30.0, random.Random(42)))
np.round(mine, 2)

## 2. Why jitter: the thundering herd

The IB Gateway restarts every night. Suppose 500 processes (yours, or a platform's) lose the connection at once and the gateway accepts at most 50 new connections per second. Without jitter, the failures stay **in lockstep**: every retry wave hits at the same instant.

In [ ]:
plain, t_plain = p.herd(jitter=False)
jit, t_jit = p.herd(jitter=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(60) - 0.2, plain[:60], width=0.4, label=f"no jitter (all in after {t_plain} s)")
ax.bar(np.arange(60) + 0.2, jit[:60], width=0.4, label=f"full jitter (all in after {t_jit} s)")
ax.axhline(50, color=p.PALETTE[7], ls="--", lw=1, label="gateway capacity")
ax.set(xlabel="seconds after the restart", ylabel="connection attempts", title="500 clients reconnecting")
ax.legend(); plt.show()
print(f"attempts in total: no jitter {plain.sum():.0f}, full jitter {jit.sum():.0f}")

## 3. The circuit breaker

Back-off handles a blip. When a broker is really down, stop calling it for a while:

* **closed**: calls go through; `threshold` consecutive failures open the breaker;
* **open**: calls are blocked until `cooldown` seconds have passed since it opened;
* **half_open**: one trial call is allowed; success closes the breaker, failure opens it again.

`record(ok, now)` is written for you. Write `allow(now)`: when open, go to half_open and allow the call once the cooldown has passed, else block it.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyBreaker(p.CircuitBreaker):
    def allow(self, now: float) -> bool:
        if self.state == "open":
            ...                                   # ✍️ cooldown passed → state "half_open", return True; else return False
        return True

outage = [(0, True), (5, False), (6, False), (7, False), (8, True), (20, True), (37, False), (40, True), (70, True), (71, True)]
mine = p.breaker_trace(MyBreaker(threshold=3, cooldown=30), outage)
mine = p.check("circuit breaker", mine, p.breaker_trace(p.CircuitBreaker(threshold=3, cooldown=30), outage))
pd.DataFrame(mine, columns=["t", "call", "state after"])

The same breaker over a 10-minute outage, with a call attempted every 5 seconds. Count how many calls actually hit the dead broker.

In [ ]:
times = np.arange(0, 900, 5)
up = (times < 120) | (times >= 720)                     # down from 2:00 to 12:00
trace = p.breaker_trace(p.CircuitBreaker(threshold=3, cooldown=60), list(zip(times, up)))
level = {"closed": 0, "half_open": 1, "open": 2}
fig, ax = plt.subplots(figsize=(10, 3))
ax.step(times / 60, [level[s] for _, _, s in trace], where="post")
ax.axvspan(2, 12, color=p.PALETTE[7], alpha=0.1, label="broker down")
ax.set(yticks=[0, 1, 2], yticklabels=list(level), xlabel="minutes", title="Circuit breaker state"); ax.legend(); plt.show()
calls_while_down = sum(1 for (t, c, _), u in zip(trace, up) if c == "call" and not u)
print(f"calls to the dead broker: {calls_while_down} with the breaker, {(~up).sum()} without it")

## 4. IB message codes

IB sends informational messages, connectivity warnings and real errors through the **same** error callback. Treating them all as errors floods your alerts; ignoring them all misses a lost connection.

| Category | Codes | Action |
|---|---|---|
| info | 2104, 2106, 2107, 2108, 2158 | log at debug ("market data farm OK") |
| connectivity | 1100, 1101, 1102, 2110, 504 | pause trading, reconnect, resubscribe |
| pacing | 162, 420 | slow down (notebook 03) |
| client_id_in_use | 326 | fatal: exit and fix the config (notebook 01) |
| order_reject | 103, 110, 201 | release the order, alert |
| unknown | anything else | log at warning and review |

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
CATEGORIES = {
    "info": {2104, 2106, 2107, 2108, 2158},
    "connectivity": {1100, 1101, 1102, 2110, 504},
    "pacing": ...,                                # ✍️ the pacing codes
    "client_id_in_use": ...,                      # ✍️ the duplicate client id code
    "order_reject": ...,                          # ✍️ the three order reject codes
}

def classify(code: int) -> str:
    for category, codes in CATEGORIES.items():
        if code in codes:
            return category
    return "unknown"

log = p.ib_error_log()
mine = [p.attempt(classify, c) for c in log]
mine = p.check("classify IB codes", mine, [p.classify_ib_code(c) for c in log])
pd.Series(mine).value_counts()

Three quarters of what arrives through the error callback is harmless information. That's why a monitor that alerts on every message gets muted by week two.

## Wrap-up

* Exponential back-off with **full jitter** for reconnects; a circuit breaker for real outages.
* Classify every broker message and give each category one action.
* Graded version: `labs/part04/week15_orders` (`backoff_delays`, `CircuitBreaker` with a fake clock, `classify_ib_code`).